# Final analysis: HD 137010 b

Fresh comparison with **50 walkers and 50,000 steps per walker**. This notebook compares three eccentricity priors and three MCMC move settings, then examines convergence by parameter, correlations, and how eccentricity/omega combinations relate to orbital period.

There are nine ensemble fits in total (22.5 million walker-steps). This may take a long time. Checkpoints are saved every 10,000 steps so interrupted runs can resume. A fixed step count is not proof of convergence.

## Plan

The transit model and likelihood are imported from the existing `src/analysis.py`. The prior comparison uses circular (`e=0`), low-eccentricity (truncated exponential, scale 0.06), and broad uniform eccentricity (`0 <= e < 0.8`). All three use the same mixture: 80% DEMove + 20% DESnookerMove.

For the move comparison, the broad-eccentricity model uses StretchMove, WalkMove, DEMove, DESnookerMove, KDEMove, GaussianMove, and the DE/Snooker mixture. Data, priors, walkers, and steps stay fixed. GaussianMove uses a diagonal covariance based on the starting proposal scales. This analysis is conditional on observing a transit, not an intrinsic planet-population model.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import emcee, corner

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "analysis.py").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "src" / "analysis.py").is_file(), "Open this notebook inside the project repository."
sys.path.insert(0, str(ROOT / "src"))
import analysis as model

WALKERS, STEPS, BURN = 50, 50_000, 0.5
SEED = 137010
OUT = ROOT / "results" / "final_50walkers_50000steps"
OUT.mkdir(parents=True, exist_ok=True)
print("Repository:", ROOT, "| Results:", OUT, "|", WALKERS, "walkers x", STEPS, "steps")

## Run settings and checkpoints

Each MCMC step advances the ensemble once. The code saves the chain and random-number state every 10,000 steps. Rerunning an interrupted cell resumes from its checkpoint; completed chains are loaded instead of overwritten.

In [ ]:
def initial_positions(kind, seed):
    rng = np.random.default_rng(seed)
    center = np.array([np.log(350), .25, np.log(.0137), 0., .462, .56, .10]
                      + ([0., 0.] if kind != "circular" else []))
    spread = np.array([.15, .15, .03, .004, .03, .04, .04]
                      + ([.2, .2] if kind != "circular" else []))
    post = model.Posterior(kind)
    pos = []
    while len(pos) < WALKERS:
        x = center + spread * rng.normal(size=len(center))
        if np.isfinite(post(x)):
            pos.append(x)
    return np.asarray(pos), post, spread

def get_move(name, spread):
    if name == "stretch": return emcee.moves.StretchMove()
    if name == "walk": return emcee.moves.WalkMove()
    if name == "de": return emcee.moves.DEMove()
    if name == "snooker": return emcee.moves.DESnookerMove()
    if name == "kde": return emcee.moves.KDEMove()
    if name == "gaussian": return emcee.moves.GaussianMove(cov=np.diag(spread**2))
    if name == "de_snooker":
        return [(emcee.moves.DEMove(), .8), (emcee.moves.DESnookerMove(), .2)]
    raise ValueError(name)

RUNS = [("circular","de_snooker"), ("lowe","de_snooker"),
        ("uniform","de_snooker"), ("uniform","stretch"), ("uniform","walk"),
        ("uniform","de"), ("uniform","snooker"), ("uniform","kde"),
        ("uniform","gaussian")]

def run_one(kind, move, run_id):
    key = f"{kind}__{move}"
    checkpoint = OUT / f"{key}_checkpoint.npz"
    init_seed = SEED + {"circular":0, "lowe":1, "uniform":2}[kind]
    positions, post, spread = initial_positions(kind, init_seed)
    ndim = positions.shape[1]
    if WALKERS < 2 * ndim:
        raise ValueError(f"Need at least {2*ndim} walkers for {ndim} parameters.")
    sampler = emcee.EnsembleSampler(WALKERS, ndim, post, moves=get_move(move, spread))
    old = np.empty((0, WALKERS, ndim)); old_lp = np.empty((0, WALKERS))
    old_acc = np.zeros(WALKERS)
    if checkpoint.exists():
        with np.load(checkpoint, allow_pickle=True) as z: # file generated by this notebook
            old, old_lp, old_acc = z["chain"], z["log_probability"], z["acceptance"]
            rng_state = tuple(z["random_state"])
    if len(old) >= STEPS:
        chain, logp, acc = old, old_lp, old_acc
        print(key, "loaded", len(chain), "steps")
    else:
        chunks = []
        if len(old):
            sampler.random_state = rng_state
            state = emcee.State(old[-1], log_prob=old_lp[-1], random_state=rng_state)
        else:
            sampler.random_state = np.random.RandomState(SEED + run_id).get_state()
            state = positions
        left = STEPS - len(old)
        for n in range(0, left, 10_000):
            block = min(10_000, left - n)
            state = sampler.run_mcmc(state if n == 0 else None, block, progress=True)
            chunks.append(sampler.get_chain()[-block:])
            chain = np.concatenate([old, *chunks], axis=0)
            logp = np.concatenate([old_lp, sampler.get_log_prob()], axis=0)
            new_n = sum(len(x) for x in chunks)
            acc = (old_acc * len(old) + sampler.acceptance_fraction * new_n) / len(chain)
            np.savez_compressed(checkpoint, chain=chain, log_probability=logp,
                acceptance=acc, random_state=np.asarray(sampler.random_state, dtype=object))
            print("Saved", key, len(chain), "/", STEPS)
    retained = chain[int(len(chain)*BURN):]
    diag = model.diagnostics(retained, acc)
    diag.update(model=kind, move=move, seed=SEED+run_id, walkers=WALKERS,
                steps=len(chain), sigma_flux=post.sigma, pmax=2000., initial_positions_seed=init_seed)
    (OUT / f"{key}_diagnostics.json").write_text(json.dumps(diag, indent=2))
    np.savez_compressed(OUT / f"{key}_posterior.npz", samples=retained.reshape(-1, ndim))
    print(key, "acceptance", round(diag["acceptance_mean"],3),
          "| convergence screen:", diag["convergence_screen_passed"])
    return chain, diag, post

In [ ]:
# Same DE/Snooker mixture for prior sensitivity; compare moves on the same broad-e model.
chains, diagnostics, posteriors = {}, {}, {}
for i, (kind, move) in enumerate(RUNS):
    key = f"{kind}__{move}"
    chains[key], diagnostics[key], posteriors[key] = run_one(kind, move, i)

## Convergence by parameter and move

The integrated autocorrelation time tau estimates how many adjacent steps behave like one independent sample. Lower tau means faster mixing for that parameter. Approximate effective sample size (ESS) is the retained number of walker-steps divided by tau.

Walkers in an ensemble interact, so the split-Rhat included here is only descriptive. Passing a screen is not proof of convergence. If tau is large, traces have not settled, or a screen fails, extend the run or use independent ensembles before making a strong claim.

In [ ]:
moves = ["uniform__stretch", "uniform__walk", "uniform__de", "uniform__snooker",
         "uniform__kde", "uniform__gaussian", "uniform__de_snooker"]
labels = model.LABELS
rows = []
for key in moves:
    d = diagnostics[key]
    for j, name in enumerate(labels):
        tau = d["tau"][j]
        rows.append({"move":key.split("__")[1], "parameter":name, "tau_steps":tau,
                     "approx_ESS":d["approx_ess"][j],
                     "retained_steps_per_tau":d["retained_steps"]/tau if np.isfinite(tau) and tau>0 else np.nan,
                     "descriptive_split_Rhat":d["split_rhat_descriptive"][j],
                     "screen_passed":d["convergence_screen_passed"]})
diagnostic_table = pd.DataFrame(rows)
display(diagnostic_table.round(2))

fig, ax = plt.subplots(figsize=(11,5))
for key in moves:
    ax.plot(labels, diagnostics[key]["tau"], marker="o", label=key.split("__")[1])
ax.set_yscale("log"); ax.set_ylabel("Integrated autocorrelation time (steps)")
ax.set_title("Mixing speed by sampled parameter"); ax.tick_params(axis="x", rotation=35)
ax.legend(); fig.tight_layout()
fig.savefig(OUT / "move_autocorrelation_by_parameter.png", dpi=180)
plt.show()

# Trace summaries for period, impact parameter, and eccentricity.
fig, axes = plt.subplots(3,1,figsize=(11,8),sharex=True)
pretty = {"uniform__stretch":"Stretch", "uniform__walk":"Walk",
          "uniform__de":"DE", "uniform__snooker":"DE-Snooker",
          "uniform__kde":"KDE", "uniform__gaussian":"Gaussian",
          "uniform__de_snooker":"80% DE + 20% Snooker"}
for key in moves:
    c=chains[key]
    vals=[c[:,:,0], c[:,:,1], c[:,:,7]**2+c[:,:,8]**2]
    for ax,v,label in zip(axes,vals,["ln(P)","Impact parameter b","Eccentricity e"]):
        step=np.arange(len(v))
        ax.plot(step,np.median(v,axis=1),label=pretty[key],lw=.8)
        ax.fill_between(step,np.percentile(v,16,axis=1),np.percentile(v,84,axis=1),alpha=.08)
        ax.set_ylabel(label)
axes[-1].set_xlabel("MCMC step"); axes[0].legend(ncol=3,fontsize=8)
fig.suptitle("Trace summaries: do the ensembles settle and overlap?")
fig.tight_layout(); fig.savefig(OUT / "move_trace_comparison.png",dpi=180); plt.show()

## Effect of the eccentricity prior

The intervals below are central 68% intervals: the 16th, median (50th), and 84th percentiles. They describe uncertainty given each model and prior.

In [ ]:
prior_keys=["circular__de_snooker","lowe__de_snooker","uniform__de_snooker"]
summary_rows=[]; physical={}
for key in prior_keys:
    kind=key.split("__")[0]
    c=chains[key][int(STEPS*BURN):].reshape(-1,chains[key].shape[-1])
    p=np.exp(c[:,0]); e=c[:,7]**2+c[:,8]**2 if c.shape[1]==9 else np.zeros(len(c))
    physical[key]={"chain":c,"period":p,"eccentricity":e}
    for label,v in [("Period [days]",p),("Eccentricity",e),("Impact parameter b",c[:,1])]:
        q16,q50,q84=np.percentile(v,[16,50,84])
        summary_rows.append({"model":kind,"quantity":label,"q16":q16,"median":q50,
                             "q84":q84,"central_68_width":q84-q16})
summary=pd.DataFrame(summary_rows); display(summary.round(3))
summary.to_csv(OUT/"final_summary.csv",index=False)

colors={"circular":"#2864a0","lowe":"#269a81","uniform":"#cc6b39"}
fig,axes=plt.subplots(1,2,figsize=(10,4))
for key in prior_keys:
    name=key.split("__")[0]; d=physical[key]
    axes[0].hist(d["period"],bins=np.geomspace(max(80,d["period"].min()),d["period"].max(),70),
                 density=True,histtype="step",lw=1.5,color=colors[name],label=name)
    axes[1].hist(d["eccentricity"],bins=np.linspace(0,.8,60),density=True,
                 histtype="step",lw=1.5,color=colors[name],label=name)
axes[0].set_xscale("log"); axes[0].set_xlabel("Period [days]"); axes[0].set_ylabel("Posterior density")
axes[0].set_title("Period posterior"); axes[1].set_xlabel("Eccentricity e")
axes[1].set_title("Eccentricity posterior"); axes[0].legend(title="Prior model")
fig.tight_layout(); fig.savefig(OUT/"prior_sensitivity_final.png",dpi=180); plt.show()

## Corner plot and correlations

Diagonal panels show single-parameter distributions. Off-diagonal panels show parameter pairs: a tilted narrow oval suggests correlation, a curved band suggests a nonlinear relation, and a round cloud suggests a weaker relation. These shapes show allowed combinations; they do not prove cause and effect.

Omega is an angle, so a group crossing +180/-180 degrees may be continuous across the plot edge.

In [ ]:
c=physical["uniform__de_snooker"]["chain"]
e=c[:,7]**2+c[:,8]**2
omega=np.degrees(np.arctan2(c[:,8],c[:,7]))
period=np.exp(c[:,0])
vals=np.column_stack([period,c[:,1],np.exp(c[:,2]),e,omega])
labels_corner=["P [days]","b","Rp/R*","e","omega [deg]"]
rng=np.random.default_rng(SEED)
take=rng.choice(len(vals),min(15000,len(vals)),replace=False)
fig=corner.corner(vals[take],labels=labels_corner,quantiles=[.16,.5,.84],
                  show_titles=True,title_fmt=".3g",plot_datapoints=False,
                  fill_contours=True,smooth=.8)
fig.suptitle("Broad-eccentricity posterior: parameter correlations",y=1.02)
fig.savefig(OUT/"corner_broad_eccentricity.png",dpi=180,bbox_inches="tight"); plt.show()

# Use sine/cosine for omega; raw degrees are not a linear variable.
cv=np.column_stack([np.log10(period),c[:,1],np.exp(c[:,2]),e,
                    np.sin(np.radians(omega)),np.cos(np.radians(omega))])
cl=["log10 P","b","Rp/R*","e","sin omega","cos omega"]
corr=np.corrcoef(cv,rowvar=False)
fig,ax=plt.subplots(figsize=(7,6)); im=ax.imshow(corr,vmin=-1,vmax=1,cmap="coolwarm")
ax.set_xticks(range(len(cl)),cl,rotation=45,ha="right"); ax.set_yticks(range(len(cl)),cl)
for i in range(len(cl)):
    for j in range(len(cl)): ax.text(j,i,f"{corr[i,j]:.2f}",ha="center",va="center",fontsize=8)
fig.colorbar(im,ax=ax,label="Pearson correlation")
ax.set_title("Linear correlations (angle represented by sine and cosine)")
fig.tight_layout(); fig.savefig(OUT/"correlation_matrix_final.png",dpi=180); plt.show()

## Which e and omega combinations go with similar periods?

Each point is a posterior sample. Its position is omega and eccentricity; its colour is the period. Similar colours indicate similar periods in this fit. Other fitted parameters also vary, so this is a view of the joint posterior, not a one-to-one formula using only e and omega.

In [ ]:
from matplotlib.colors import LogNorm
fig,ax=plt.subplots(figsize=(9,5))
take=rng.choice(len(e),min(30000,len(e)),replace=False)
sc=ax.scatter(omega[take],e[take],c=period[take],s=5,alpha=.45,cmap="viridis",
              norm=LogNorm(vmin=max(90,period[take].min()),vmax=period[take].max()))
ax.set_xlabel("Argument of periastron omega [deg]"); ax.set_ylabel("Eccentricity e")
ax.set_title("Posterior e and omega combinations, coloured by orbital period")
fig.colorbar(sc,ax=ax,label="Orbital period [days], logarithmic colour scale")
fig.tight_layout(); fig.savefig(OUT/"eccentricity_omega_period_final.png",dpi=180); plt.show()

## Transit checks and presentation figures

The existing project plotting function below uses these fresh chains to save transit fits, residual checks, representative solutions, derived quantities, and the circular comparison with supplied posterior samples. That circular comparison remains approximate because the likelihood and nuisance treatment differ from the paper's full analysis.

In [ ]:
prior_runs={}
for key in prior_keys:
    kind=key.split("__")[0]
    retained=chains[key][int(STEPS*BURN):]
    flat=retained.reshape(-1,retained.shape[-1])
    prior_runs[kind]=(flat,retained,diagnostics[key],posteriors[key])
model.make_outputs(prior_runs,OUT,SEED)
print("Figures and tables saved in:",OUT)

## What to report

- Give every result with its prior and move setting.
- Report the median and central 68% interval (16th to 84th percentile).
- Compare autocorrelation times by parameter; do not declare a move best from one run alone.
- Use the corner plot to describe correlations and nonlinear degeneracies.
- Similar colours in the e-omega plot mean similar periods, but other parameters also contribute.
- If the convergence screen fails, call the result exploratory and extend the chain before claiming a stable posterior.